# 09 — 整图分类：从 global pooling 到 Graph Transformer

Cora 是一个大图中的节点分类；MUTAG 则包含许多大小不同的分子图，目标是为每张图预测类别。本课学习 mini-batch、`batch` 向量、global pooling、inductive split，并比较 GraphMLP、GCN、GIN 与 GPS Graph Transformer。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch_geometric.loader import DataLoader

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.graph_classification import load_tu_dataset, stratified_graph_split
from crosscity.models.graph_classification import GraphMLP, GraphGCN, GraphGIN, GraphGPS
from crosscity.training.graph_classification import train_graph_classifier
from crosscity.utils import seed_everything
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. 下载并检查 MUTAG

节点表示原子，边表示化学键，节点特征包含原子类别。PyG 首次运行会下载并缓存数据。注意 MUTAG 只有 188 张图，适合教学但不适合仅凭一次划分宣称模型优劣。

In [ ]:
dataset = load_tu_dataset(ROOT / 'data/raw/tu', 'MUTAG')
splits = stratified_graph_split(dataset, seed=42)
print('graphs/features/classes:', len(dataset), dataset.num_features, dataset.num_classes)
print('train/validation/test:', len(splits.train), len(splits.validation), len(splits.test))
print('first graph:', dataset[0])

## 2. mini-batch 不是补零矩阵

PyG 把多张图拼成一个互不连通的大图，并用 `batch[i]` 标记第 i 个节点属于哪张图。`global_mean_pool(x, batch)` 再把节点表示还原成每图一个向量。

In [ ]:
inspect_batch = next(iter(DataLoader(splits.train, batch_size=4, shuffle=False)))
print(inspect_batch)
print('nodes per graph:', torch.bincount(inspect_batch.batch).tolist())
print('batch vector head:', inspect_batch.batch[:30].tolist())
assert int(inspect_batch.edge_index.max()) < inspect_batch.num_nodes

## 3. 四种模型

- GraphMLP：忽略化学键，独立编码原子后求均值；
- GCN：度归一化的局部消息传递；
- GIN：可学习的 sum aggregation，理论上具有更强的图结构区分能力；
- GPS：每层同时进行局部 GIN 和图内全局多头自注意力。

GPS 的 attention 只在同一张图内部进行，`batch` 防止不同分子的节点互相注意。

In [ ]:
train_loader = DataLoader(splits.train, batch_size=32, shuffle=True)
validation_loader = DataLoader(splits.validation, batch_size=64)
test_loader = DataLoader(splits.test, batch_size=64)
models = {'GraphMLP': GraphMLP, 'GCN': GraphGCN, 'GIN': GraphGIN, 'GPS': GraphGPS}
for name, model_class in models.items():
    seed_everything(42)
    model = model_class(splits.num_features, 64, splits.num_classes)
    result = train_graph_classifier(model, train_loader, validation_loader, test_loader,
                                    device=device, max_epochs=5, patience=5)
    print(name, 'val=', round(result.validation_accuracy, 3),
          'output shape=', tuple(model(next(iter(test_loader)).to(device).x,
                                      next(iter(test_loader)).to(device).edge_index,
                                      next(iter(test_loader)).to(device).batch).shape))

## 4. 三种子正式比较

候选模型和数据划分已经固定。只用 validation 做早停；test 仅用于最终确认。这里变化的是模型初始化和训练顺序，数据 split 始终保持 seed=42。

In [ ]:
records, histories = [], {}
for seed in [42, 43, 44]:
    for name, model_class in models.items():
        seed_everything(seed)
        train_loader = DataLoader(splits.train, batch_size=32, shuffle=True,
                                  generator=torch.Generator().manual_seed(seed))
        model = model_class(splits.num_features, 64, splits.num_classes)
        result = train_graph_classifier(model, train_loader, validation_loader, test_loader,
                                        device=device, max_epochs=300, patience=50)
        records.append({'seed': seed, 'model': name, 'best_epoch': result.best_epoch,
                        'validation_accuracy': result.validation_accuracy,
                        'test_accuracy': result.test_accuracy})
        histories[(seed, name)] = result.history
results = pd.DataFrame(records)
display(results.sort_values(['seed', 'validation_accuracy'], ascending=[True, False]))
summary = results.groupby('model')[['validation_accuracy', 'test_accuracy']].agg(['mean', 'std'])
summary.style.format('{:.3f}')

In [ ]:
for (seed, name), history in histories.items():
    if seed == 42:
        frame = pd.DataFrame(history)
        plt.plot(frame.epoch, frame.val_accuracy, label=name)
plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.legend(); plt.show()

## 5. 解释清单

1. GNN 是否稳定超过 GraphMLP？这回答化学键是否提供原子组成之外的信息。
2. GIN 是否超过 GCN？不要只看单个种子。
3. GPS 是否因全局 attention 获益，还是在小数据上更容易过拟合？
4. 每张 test 图在训练时完全不可见，因此这与 Cora 的 transductive 节点分类有何不同？
5. 下一步可加入 degree/Laplacian positional encoding，检验没有位置编码的全局 attention 能识别多少结构。